# PRODUCTION

In [1]:
### ==== IMPORT ====
import pypdf
import logging
from pathlib import Path
import pandas as pd
import re
import spacy

# Load the NLP model - with installation fallback
try:
    nlp = spacy.load("en_core_web_sm")
    print("✓ spaCy model loaded successfully!")
except OSError:
    print("⚠ spaCy model not found.")
    print("  To fix this, run in terminal: python -m spacy download en_core_web_sm")
    print("  For now, extraction will use pattern matching (reduced accuracy).")
    nlp = None


# Suppress all pypdf log messages below the ERROR level
logging.getLogger("pypdf").setLevel(logging.ERROR)

✓ spaCy model loaded successfully!


In [2]:
### ==== HELPTER FUNCTIONS ====

def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "pyproject.toml").exists() or (candidate / "README.md").exists():
            return candidate
    return start

In [3]:
### ==== FILE PATHS ====
# Resolve the repository root (folder containing .git), then point to your PDF directory.

project_root = locate_project_root(Path.cwd())
print(f"Project root: {project_root}")

# Update this relative path if your PDFs are stored elsewhere in the repo
folder_path = project_root / "Data/RAW_test/"

pdf_files = sorted(folder_path.glob("*.pdf"))

if not pdf_files:
    print(f"No PDF files found in: {folder_path}")
else:
    total_files = len(pdf_files)
    print(f"Found {total_files} PDF file(s) in: {folder_path}")

    user_input = input(f"How many files do you want to process? (1-{total_files}): ").strip()

    try:
        num_to_process = int(user_input)
        if num_to_process <= 0:
            raise ValueError
    except ValueError:
        raise SystemExit("Error: Invalid input. Please enter a positive integer. Execution terminated.")

    if num_to_process > total_files:
        choice = input(
            f"Error: Requested {num_to_process} files, but only {total_files} available.\n"
            "Type 'all' to process all files, or 'q' to terminate: "
        ).strip().lower()

        if choice == "all":
            num_to_process = total_files
        elif choice in {"q", "quit"}:
            raise SystemExit("Execution terminated by user. No files were processed.")
        else:
            raise SystemExit("Error: Invalid choice. Execution terminated without processing files.")

Project root: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR
Found 23 PDF file(s) in: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/Data/RAW_test


# DEVELOPMENT

In [4]:
### ==== FUNCTIONS ====
def extract_supervisors(reader, max_pages=10):

    supervisor_keywords = {
        "supervisor", "supervisors", "advisor", "advisors", "adviser", "advisers",
        "vejleder", "vejledere", "thesis supervisor", "project supervisor", "guided by",
        "speciale vejleder", "speciale supervisor", "supervised by"
    }
    affiliation_keywords = {
        "university", "department", "institute", "faculty", "school", "center", "centre",
        "dtu", "technical university", "danmarks tekniske universitet", 
        "technical university of denmark"
    }

    def normalize(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip(" -:\t")

    def looks_like_name(text: str) -> bool:
        text = normalize(text)
        if not text or any(ch.isdigit() for ch in text):
            return False
        tokens = re.findall(r"[A-Za-zÆØÅæøå][A-Za-zÆØÅæøå\-\.'’]*", text)
        if not (2 <= len(tokens) <= 6):
            return False
        capitalized = sum(
            1 for t in tokens
            if re.match(r"^[A-ZÆØÅ][a-zæøåA-ZÆØÅ\-\.'’]*$", t) or re.match(r"^[A-ZÆØÅ]\.$", t)
        )
        return capitalized >= 2

    candidates = []
    seen = set()

    pages_to_scan = min(max_pages, len(reader.pages))
    for page_idx in range(pages_to_scan):
        text = reader.pages[page_idx].extract_text() or ""
        lines = [normalize(line) for line in text.splitlines() if normalize(line)]

        for i, line in enumerate(lines):
            lower = line.lower()

            keyword_hit = any(k in lower for k in supervisor_keywords)
            if not keyword_hit:
                continue

            # Try extracting right side after labels like "Supervisor: John Doe"
            m = re.search(
                r"(supervisors?|advis(?:or|er)s?|vejled(?:er|ere))\s*[:\-]\s*(.+)$",
                line,
                flags=re.IGNORECASE,
            )
            if m:
                value = normalize(m.group(2))
                if value and value.lower() not in seen:
                    seen.add(value.lower())
                    candidates.append(f"p.{page_idx + 1}: {value}")

            # Include nearby context lines (name + affiliation often split across lines)
            for j in range(max(0, i - 2), min(len(lines), i + 3)):
                ctx = lines[j]
                ctx_lower = ctx.lower()

                if (
                    any(k in ctx_lower for k in supervisor_keywords)
                    or any(k in ctx_lower for k in affiliation_keywords)
                    or looks_like_name(ctx)
                ):
                    key = ctx_lower
                    if key not in seen:
                        seen.add(key)
                        candidates.append(f"p.{page_idx + 1}: {ctx}")

    return candidates

In [5]:
### ==== MAIN FUNCTION ====
# Opem and read in a LOCAL PDF file
def main(pdf_path): # , supres_prints=True):
    with open(pdf_path, 'rb') as file:
        try:
            reader = pypdf.PdfReader(file)
        except Exception as e:
            print(f"Error reading PDF file: {e}")

        suporvisors_candidates = extract_supervisors(reader, max_pages=10)
        print(f"Extracted supervisors: {suporvisors_candidates}")


    return reader


## To run program:

In [6]:
# Building dataframe to store the metrics for each PDF file
extensive_metrics_df = pd.DataFrame(columns=["pdf_file", "member_id_ss_metrics", "supervisor(s)"])

# Process the specified number of PDF files and collect metrics.
for current_file_number, pdf_path in enumerate(pdf_files[:num_to_process], start=1):
    print(f"\n=== Processing file {current_file_number} of {num_to_process} ===")
    print(f"Current file is: {pdf_path.name}\n")
    main(pdf_path)


=== Processing file 1 of 5 ===
Current file is: 5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf

Extracted supervisors: ['p.2: Advisors', 'p.2: Ole Ravn, Electrical Head of Department at DTU', 'p.2: Masashi Sugiyama, Director of RIKEN-AIP', 'p.5: I would like to thank my supervisor at DTU, Ole Ravn, for always supporting me whilst I was', 'p.5: abroad in Japan. I would also like to thank my supervisor at RIKEN-AIP, Prof Sugiyama, for']

=== Processing file 2 of 5 ===
Current file is: 5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf

Extracted supervisors: ['p.2: Universitet Danmarks Tekniske Universitet', 'p.2: Vejleder Pawel Wargocki', 'p.3: pelig vejleder under hele projektet. Derudover vil jeg gerne takke gæstestuderende Lulu og King']

=== Processing file 3 of 5 ===
Current file is: 5d

### WITH A LITTLE HELP FROM MY FRIEND

In [13]:
def extract_supervisors_refined(reader, max_pages=10):
    if nlp is None:
        print("Warning: Using pattern-based extraction (spaCy model not available)")
        return extract_supervisors_pattern_based(reader, max_pages)

    not_names = [
        "DTU",
        "technical university of denmark",
        "danmarks tekniske universitet",
        "vil jeg gerne takke",
        "I want to thank",
    ]
    not_names_normalized = {x.lower().strip() for x in not_names}

    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def is_valid_name_candidate(text: str) -> bool:
        candidate = normalize_spaces(text).strip(" ,.-:;\t")
        if not candidate:
            return False

        words = candidate.split()
        if len(words) < 2 or len(words) > 6:
            return False

        if candidate.lower() in not_names_normalized:
            return False

        return True

    # Titles to strip to get clean names
    titles_pattern = r"\b(Prof\.|Professor|Associate [Pp]rofessor|Assistant [Pp]rofessor|Ph\.D\.|PhD|MSc|BSc|Vejleder|Advisor|Supervisor|Adviser)\b"

    found_data = []
    seen_names = set()

    pages_to_scan = min(max_pages, len(reader.pages))
    for page_idx in range(pages_to_scan):
        text = reader.pages[page_idx].extract_text() or ""

        # Use spaCy only for PERSON entity extraction.
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                raw_name = ent.text.strip().replace("\n", " ")
                clean_name = re.sub(titles_pattern, "", raw_name, flags=re.IGNORECASE).strip(" ,.-")

                if not is_valid_name_candidate(clean_name):
                    continue

                if clean_name.lower() not in seen_names:
                    seen_names.add(clean_name.lower())
                    found_data.append({
                        "name": clean_name,
                        "page": page_idx + 1
                    })

    return found_data


def extract_supervisors_pattern_based(reader, max_pages=10):
    """Fallback extraction using pattern matching (simplified, less accurate)."""
    supervisor_keywords = {
        "supervisor", "supervisors", "advisor", "advisors", "adviser", "advisers",
        "vejleder", "vejledere", "thesis supervisor", "project supervisor", "guided by",
        "speciale vejleder", "speciale supervisor", "supervised by"
    }

    not_names = [
        "DTU",
        "technical university of denmark",
        "danmarks tekniske universitet",
        "vil jeg gerne takke",
        "I want to thank",
    ]
    not_names_normalized = {x.lower().strip() for x in not_names}

    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def is_valid_name_candidate(text: str) -> bool:
        candidate = normalize_spaces(text).strip(" ,.-:;\t")
        if not candidate:
            return False

        words = candidate.split()
        if len(words) < 2 or len(words) > 6:
            return False

        if candidate.lower() in not_names_normalized:
            return False

        return True

    found_data = []
    seen_names = set()

    pages_to_scan = min(max_pages, len(reader.pages))
    for page_idx in range(pages_to_scan):
        text = reader.pages[page_idx].extract_text() or ""
        lines = text.splitlines()

        for line in lines:
            line_lower = line.lower()
            if any(k in line_lower for k in supervisor_keywords):
                # Extract text after colon/dash if present.
                if ':' in line:
                    parts = line.split(':', 1)
                    candidate = parts[1].strip()
                elif '-' in line:
                    parts = line.split('-', 1)
                    candidate = parts[1].strip()
                else:
                    candidate = line

                if is_valid_name_candidate(candidate):
                    normalized = normalize_spaces(candidate).lower()
                    if normalized not in seen_names:
                        seen_names.add(normalized)
                        found_data.append({
                            "name": normalize_spaces(candidate),
                            "page": page_idx + 1
                        })

    return found_data


def extract_supervisors_hybrid(reader, max_pages=10):
    """Hybrid approach: Use pattern-based extraction, then refine with NLP."""
    not_names = [
        "DTU",
        "technical university of denmark",
        "danmarks tekniske universitet",
        "vil jeg gerne takke",
        "I want to thank",
        "Universitet Danmarks Tekniske Universitet",
        "The Technical University of Denmark",
    ]
    not_names_normalized = {x.lower().strip() for x in not_names}

    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def is_valid_name_candidate(text: str) -> bool:
        candidate = normalize_spaces(text).strip(" ,.-:;\t")
        if not candidate:
            return False

        words = candidate.split()
        if len(words) < 2 or len(words) > 6:
            return False

        if candidate.lower() in not_names_normalized:
            return False

        return True

    if nlp is None:
        print("Note: spaCy model not available. Using pattern-based extraction only.")
        return extract_supervisors_pattern_based(reader, max_pages)

    # Step 1: Get candidates from pattern-based extraction.
    pattern_candidates = extract_supervisors(reader, max_pages)

    # Step 2: Refine candidates using NLP PERSON extraction only.
    titles_pattern = r"\b(Prof\.|Professor|Associate [Pp]rofessor|Assistant [Pp]rofessor|Ph\.D\.|PhD|MSc|BSc|Vejleder|Advisor|Supervisor|Adviser)\b"

    refined_data = []
    seen_names = set()

    for candidate_text in pattern_candidates:
        # Extract page number and candidate text (format: "p.X: text").
        parts = candidate_text.split(": ", 1)
        page_info = parts[0]
        text_to_process = parts[1] if len(parts) > 1 else candidate_text
        page_num = int(page_info.split(".")[-1])

        # Run spaCy NER on the candidate text.
        doc = nlp(text_to_process)

        person_found = False
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                raw_name = ent.text.strip().replace("\n", " ")
                clean_name = re.sub(titles_pattern, "", raw_name, flags=re.IGNORECASE).strip(" ,.-")

                if not is_valid_name_candidate(clean_name):
                    continue

                if clean_name.lower() not in seen_names:
                    seen_names.add(clean_name.lower())
                    refined_data.append({
                        "name": clean_name,
                        "page": page_num,
                        "source": "hybrid"
                    })
                    person_found = True

        # If no PERSON entity found, keep the raw candidate only if valid.
        normalized_raw = normalize_spaces(text_to_process)
        if not person_found and is_valid_name_candidate(normalized_raw):
            lowered = normalized_raw.lower()
            if lowered not in seen_names and lowered not in not_names_normalized:
                seen_names.add(lowered)
                refined_data.append({
                    "name": normalized_raw,
                    "page": page_num,
                    "source": "pattern_only"
                })

    return refined_data

In [14]:
### ==== MAIN FUNCTION ====
# Open and read in a LOCAL PDF file
def main(pdf_path): # , supres_prints=True):
    with open(pdf_path, 'rb') as file:
        try:
            reader = pypdf.PdfReader(file)
        except Exception as e:
            print(f"Error reading PDF file: {e}")
            return None

        # Use hybrid approach: pattern-based extraction refined with NLP
        supervisors_candidates = extract_supervisors_hybrid(reader, max_pages=10)
        print(f"Extracted supervisors: {supervisors_candidates}")

    return reader

In [12]:
# Building dataframe to store the metrics for each PDF file
extensive_metrics_df = pd.DataFrame(columns=["pdf_file", "member_id_ss_metrics", "supervisor(s)"])

# Process the specified number of PDF files and collect metrics.
for current_file_number, pdf_path in enumerate(pdf_files[:num_to_process], start=1):
    print(f"\n=== Processing file {current_file_number} of {num_to_process} ===")
    print(f"Current file is: {pdf_path.name}\n")
    main(pdf_path)


=== Processing file 1 of 5 ===
Current file is: 5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf

Extracted supervisors: [{'name': 'Ole Ravn', 'department': 'DTU', 'page': 2, 'source': 'hybrid'}, {'name': 'Masashi Sugiyama, Director of RIKEN-AIP', 'department': 'Not specified', 'page': 2, 'source': 'pattern_only'}]

=== Processing file 2 of 5 ===
Current file is: 5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf

Extracted supervisors: [{'name': 'Pawel Wargocki', 'department': 'Not specified', 'page': 2, 'source': 'hybrid'}, {'name': 'Lulu og King', 'department': 'Derudover', 'page': 3, 'source': 'hybrid'}]

=== Processing file 3 of 5 ===
Current file is: 5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model ind